In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.nn.utils import clip_grad_norm_
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from scipy.spatial import cKDTree
from matplotlib import pyplot as plt
import math
import os

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
MAX_LEN = 1202               # 最大序列长度（数据集中最长为1202）
D_MODEL = 128                # Transformer 隐藏维度
NHEAD = 8                    # 注意力头数
NUM_LAYERS = 8               # Transformer 层数
DIM_FEEDFORWARD = 256        # FFN 维度
BATCH_SIZE = 128             # 蛋白质样本数（显存限制）
EPOCHS = 50
LEARNING_RATE = 1e-3
DROPOUT = 0.2

K_NEIGHBORS = 16             # 局部密度计算时的邻居数
LOCAL_RADIUS = 10.0          # 局部密度半径 (Å)

# 路径
TRAIN_CSV = "train.csv"
A_TEST_CSV = "A.csv"
B_TEST_CSV = "B.csv"
OUTPUT_A = "A_predict.csv"
OUTPUT_B = "B_predict.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 加载训练集（仅用于 EDA）
train_df = pd.read_csv(TRAIN_CSV)
print(f"训练集蛋白质数量: {len(train_df)}")

# 氨基酸频率统计（基于第一个样本的标签演示）
def parse_labels(labels_str):
    return np.array([int(x) for x in labels_str.split(',')])

all_labels = []
for row in train_df.iterrows():
    labels = parse_labels(row[1]['labels'])
    all_labels.extend(labels)
label_counts = pd.Series(all_labels).value_counts().sort_index()
print("氨基酸类别分布（0~19）:")
print(label_counts)

# 长度分布
lengths = [len(parse_labels(row[1]['labels'])) for row in train_df.iterrows()]
plt.figure(figsize=(10,4))
plt.hist(lengths, bins=50, alpha=0.7)
plt.xlabel("Protein length (residues)")
plt.ylabel("Frequency")
plt.title("Distribution of protein lengths in training set")
plt.show()

# 坐标范围（第一个蛋白质的示例）
def parse_coords(coords_str):
    values = np.array([float(x) for x in coords_str.split(',')])
    return values.reshape(-1, 3)

sample_coords = parse_coords(train_df.iloc[0]['coords'])
print(f"Sample protein coordinates: shape {sample_coords.shape}")
print(f"X range: [{sample_coords[:,0].min():.2f}, {sample_coords[:,0].max():.2f}]")
print(f"Y range: [{sample_coords[:,1].min():.2f}, {sample_coords[:,1].max():.2f}]")
print(f"Z range: [{sample_coords[:,2].min():.2f}, {sample_coords[:,2].max():.2f}]")

In [ ]:
def parse_coords(coords_str):
    values = np.array([float(x) for x in coords_str.split(',')])
    return values.reshape(-1, 3)

def parse_labels(labels_str):
    return np.array([int(x) for x in labels_str.split(',')])

def compute_geometry_features(coords):
    """
    为每个残基计算几何特征，返回 (L, feat_dim) 数组。
    特征包括：
      1. 自身坐标（3维，标准化前保留，后面会整体标准化）
      2. 到前后残基的距离（2维）
      3. 到前后残基的方向向量（2*3=6维）
      4. 到质心的距离（1维）
      5. 局部密度（半径10Å内残基数，1维）
      6. 到最近残基的距离（1维）
      7. 局部曲率：三个连续残基的夹角（1维）
    """
    L = len(coords)
    if L < 3:
        # 太短的蛋白质，简单处理
        features = np.zeros((L, 1+1+1+1))  # 仅保留少量特征
        # 但为了统一，我们仍返回固定维度，下面按实际情况实现
    # 初始化特征数组
    feat_list = []
    
    # 质心
    centroid = np.mean(coords, axis=0)
    
    # 预先计算所有残基之间的距离矩阵（用于密度和最近邻）
    dist_matrix = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    # 自身距离设为 inf
    np.fill_diagonal(dist_matrix, np.inf)
    
    for i in range(L):
        # 1. 自身坐标（先保留，后面统一标准化）
        xyz = coords[i]
        # 2. 到前后残基的距离
        dist_prev = np.linalg.norm(coords[i] - coords[i-1]) if i > 0 else 0.0
        dist_next = np.linalg.norm(coords[i] - coords[i+1]) if i < L-1 else 0.0
        # 3. 到前后残基的方向向量
        dir_prev = (coords[i-1] - coords[i]) / (dist_prev + 1e-8) if i > 0 else np.zeros(3)
        dir_next = (coords[i+1] - coords[i]) / (dist_next + 1e-8) if i < L-1 else np.zeros(3)
        # 4. 到质心的距离
        dist_center = np.linalg.norm(xyz - centroid)
        # 5. 局部密度（半径内残基数，不包括自身）
        density = np.sum(dist_matrix[i] < LOCAL_RADIUS)
        # 6. 到最近残基的距离
        min_dist = np.min(dist_matrix[i])
        # 7. 局部曲率：使用 i-1, i, i+1 三点夹角（如果存在）
        angle = 0.0
        if 0 < i < L-1:
            v1 = coords[i-1] - coords[i]
            v2 = coords[i+1] - coords[i]
            cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
            angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        
        feat = np.concatenate([
            xyz,                           # 3
            [dist_prev, dist_next],        # 2
            dir_prev, dir_next,            # 6
            [dist_center],                 # 1
            [density],                     # 1
            [min_dist],                    # 1
            [angle]                        # 1
        ])
        feat_list.append(feat)
    
    features = np.array(feat_list)  # (L, 3+2+6+1+1+1+1 = 15)
    return features

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, csv_path, feature_func, label_col=None):
        self.df = pd.read_csv(csv_path)
        self.feature_func = feature_func
        self.label_col = label_col
        self.samples = []
        for _, row in self.df.iterrows():
            coords = parse_coords(row['coords'])
            if len(coords) == 0:
                continue
            features = self.feature_func(coords)   # (L, feat_dim)
            data = {'coords': coords, 'features': features, 'domain_id': row['domain_id']}
            if label_col and label_col in row:
                labels = parse_labels(row[label_col])
                # 确保长度一致（可能因特征计算导致长度微调？但特征与坐标一一对应）
                min_len = min(len(features), len(labels))
                data['features'] = features[:min_len]
                data['labels'] = labels[:min_len]
                data['length'] = min_len
            else:
                data['length'] = len(features)
            self.samples.append(data)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    """padding 变长序列，并生成 attention_mask"""
    max_len = max(sample['length'] for sample in batch)
    feat_dim = batch[0]['features'].shape[1]
    batch_features = []
    batch_labels = []
    batch_mask = []
    batch_domain_ids = []
    for sample in batch:
        L = sample['length']
        # features padding
        feat = sample['features']
        pad_len = max_len - L
        if pad_len > 0:
            feat_pad = np.zeros((pad_len, feat_dim), dtype=np.float32)
            feat = np.vstack([feat, feat_pad])
        batch_features.append(feat)
        # mask: 1 表示有效，0 表示 padding
        mask = np.zeros(max_len, dtype=np.float32)
        mask[:L] = 1.0
        batch_mask.append(mask)
        batch_domain_ids.append(sample['domain_id'])
        if 'labels' in sample:
            labels = sample['labels']
            pad_len_labels = max_len - len(labels)
            if pad_len_labels > 0:
                labels = np.concatenate([labels, np.zeros(pad_len_labels, dtype=labels.dtype)])
            batch_labels.append(labels)
    features = torch.tensor(np.array(batch_features), dtype=torch.float32)
    mask = torch.tensor(np.array(batch_mask), dtype=torch.float32)
    if batch_labels:
        labels = torch.tensor(np.array(batch_labels), dtype=torch.long)
        return features, mask, labels, batch_domain_ids
    else:
        return features, mask, batch_domain_ids

In [ ]:
# 构建训练集，提取所有特征用于标准化
import pickle

# 自定义 Dataset 用于从保存的样本列表加载
class ProteinDatasetFromList(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

# 检查缓存是否存在
if os.path.exists("train_samples.pkl"):
    print("Loading cached samples...")
    with open("train_samples.pkl", "rb") as f:
        samples = pickle.load(f)
    train_dataset_raw = ProteinDatasetFromList(samples)
else:
    print("Computing features from scratch...")
    train_dataset_raw = ProteinDataset(TRAIN_CSV, compute_geometry_features, label_col='labels')
    # 保存样本列表
    with open("train_samples.pkl", "wb") as f:
        pickle.dump(train_dataset_raw.samples, f)
    print("Saved train_samples.pkl")

if os.path.exists("all_features.npy"):
    all_feats = np.load("all_features.npy")
else:
    # 收集所有特征向量（用于计算均值和标准差）
    all_feats = []
    for sample in train_dataset_raw.samples:
        all_feats.append(sample['features'])

    all_feats = np.vstack(all_feats)

    np.save("all_features.npy", all_feats)

scaler = StandardScaler()
scaler.fit(all_feats)
print(f"特征维度: {all_feats.shape[1]}")
print(f"总残基数: {len(all_feats)}")

# 保存标准化器，用于测试集
import joblib
joblib.dump(scaler, "feature_scaler.pkl")

In [ ]:
def collate_fn(batch):
    """
    batch: list of dict, each dict contains:
        - 'features': (L, feat_dim)
        - 'labels': (L,) (if training)
        - 'domain_id': str
        - 'length': int
    """
    max_len = max(sample['length'] for sample in batch)
    feat_dim = batch[0]['features'].shape[1]
    batch_features = []
    batch_labels = []
    batch_mask = []
    batch_domain_ids = []
    
    for sample in batch:
        L = sample['length']
        feat = sample['features']
        pad_len = max_len - L
        if pad_len > 0:
            feat_pad = np.zeros((pad_len, feat_dim), dtype=np.float32)
            feat = np.vstack([feat, feat_pad])
        batch_features.append(feat)
        
        # mask: 1 for valid, 0 for padding
        mask = np.zeros(max_len, dtype=np.float32)
        mask[:L] = 1.0
        batch_mask.append(mask)
        batch_domain_ids.append(sample['domain_id'])
        
        if 'labels' in sample:
            labels = sample['labels']
            pad_len_labels = max_len - len(labels)
            if pad_len_labels > 0:
                labels = np.concatenate([labels, np.zeros(pad_len_labels, dtype=labels.dtype)])
            batch_labels.append(labels)
    
    features = torch.tensor(np.array(batch_features), dtype=torch.float32)
    mask = torch.tensor(np.array(batch_mask), dtype=torch.float32)
    
    if batch_labels:
        labels = torch.tensor(np.array(batch_labels), dtype=torch.long)
        return features, mask, labels, batch_domain_ids
    else:
        return features, mask, batch_domain_ids

In [ ]:
def apply_scaler(sample):
    sample['features'] = scaler.transform(sample['features'])
    return sample

train_samples = [apply_scaler(s) for s in train_dataset_raw.samples]

# 划分训练/验证集（按蛋白质样本划分）
train_samples, val_samples = train_test_split(train_samples, test_size=0.1, random_state=42)

train_dataset = ProteinDataset.__new__(ProteinDataset)
train_dataset.samples = train_samples
val_dataset = ProteinDataset.__new__(ProteinDataset)
val_dataset.samples = val_samples

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=4)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

class ProteinTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dim_feedforward, dropout, num_classes=20):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, MAX_LEN)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )
    
    def forward(self, x, mask):
        """
        x: (batch, seq_len, input_dim)
        mask: (batch, seq_len) 1 for valid, 0 for padding
        """
        # 转换到 d_model
        x = self.input_proj(x)   # (batch, seq_len, d_model)
        x = self.pos_encoder(x)
        # Transformer 需要 key_padding_mask: (batch, seq_len), True 表示需要被 mask 的位置
        key_padding_mask = (mask == 0)   # True for padding
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        logits = self.output_proj(x)
        return logits

input_dim = train_dataset_raw.samples[0]['features'].shape[1]
model = ProteinTransformer(input_dim, D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD, DROPOUT, num_classes=20).to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_acc, model, model_path="best_model.pth"):
        if self.best_score is None:
            self.best_score = val_acc
            self.save_checkpoint(model, model_path)
        elif val_acc < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.restore_best_weights:
                    model.load_state_dict(torch.load(model_path))
                print(f"Early stopping triggered after {self.counter} epochs without improvement.")
        else:
            self.best_score = val_acc
            self.save_checkpoint(model, model_path)
            self.counter = 0

    def save_checkpoint(self, model, model_path):
        torch.save(model.state_dict(), model_path)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# 初始化 EarlyStopping
early_stopping = EarlyStopping(patience=5, min_delta=0.001, restore_best_weights=True)
optimizer = optim.RAdam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-5)

# 计算类别权重（基于训练集标签频率）
class_counts = np.bincount(all_labels)  # all_labels 是训练集所有残基标签
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * 20
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1, reduction='none')

best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    for features, mask, labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Train", leave=False):
        features = features.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(features, mask)
        ce = criterion(logits.permute(0, 2, 1), labels)
        n_valid = mask.sum().clamp(min=1)
        loss = (ce * mask).sum() / n_valid
        loss.backward()

        total_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), 
            max_norm=2.0, 
            norm_type=2
        )

        optimizer.step()
        total_loss += (ce * mask).sum().item()
        total_tokens += mask.sum().item()
    
    avg_loss = total_loss / total_tokens
    
    # 验证
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, mask, labels, _ in val_loader:
            features = features.to(device)
            mask = mask.to(device)
            labels = labels.to(device)
            logits = model(features, mask)
            preds = torch.argmax(logits, dim=-1)
            valid = mask.bool()
            correct += (preds[valid] == labels[valid]).sum().item()
            total += valid.sum().item()

    val_acc = correct / total
    # 调整学习率（余弦退火）
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:2d}/{EPOCHS}, Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}, LR: {current_lr:.6f}")
    
    # 更新最佳验证准确率（用于记录，可选）
    if val_acc > best_val_acc:
        best_val_acc = val_acc
    
    # EarlyStopping 检查
    early_stopping(val_acc, model, model_path="best_model.pth")
    if early_stopping.early_stop:
        print(f"Training stopped early. Best validation accuracy: {early_stopping.best_score:.4f}")
        break

# 确保加载最佳模型（EarlyStopping 可能已经恢复，但显式加载确保）
model.load_state_dict(torch.load("best_model.pth"))
print(f"Training finished. Best validation accuracy: {early_stopping.best_score:.4f}")

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

def predict_protein(coords_str, scaler, model, feature_func):
    coords = parse_coords(coords_str)
    if len(coords) == 0:
        return []
    features = feature_func(coords)
    features_scaled = scaler.transform(features)
    # 转为 batch 格式
    X = torch.tensor(features_scaled, dtype=torch.float32).unsqueeze(0).to(device)  # (1, L, dim)
    mask = torch.ones(1, len(features), dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X, mask)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()
    return preds.tolist()

In [ ]:
# 预测 A 榜
print("预测 A 榜...")
a_df = pd.read_csv(A_TEST_CSV)
a_predictions = []
for idx, row in tqdm(a_df.iterrows(), total=len(a_df), desc="A榜"):
    pred_labels = predict_protein(row['coords'], scaler, model, compute_geometry_features)
    a_predictions.append(','.join(map(str, pred_labels)))
a_out = pd.DataFrame({'domain_id': a_df['domain_id'], 'predictions': a_predictions})
a_out.to_csv(OUTPUT_A, index=False)
print(f"A 榜预测完成，保存至 {OUTPUT_A}")

In [ ]:
print("预测 B 榜...")
b_df = pd.read_csv(B_TEST_CSV)
b_predictions = []
for idx, row in tqdm(b_df.iterrows(), total=len(b_df), desc="B榜"):
    pred_labels = predict_protein(row['coords'], scaler, model, compute_geometry_features)
    b_predictions.append(','.join(map(str, pred_labels)))
b_out = pd.DataFrame({'domain_id': b_df['domain_id'], 'predictions': b_predictions})
b_out.to_csv(OUTPUT_B, index=False)
print(f"B 榜预测完成，保存至 {OUTPUT_B}")